# Click CNN Training (PyTorch)

This notebook trains a CNN classifier on images in `data/click` and saves the trained model under `models/basic_cnn`.

In [1]:
from pathlib import Path
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Resolve dataset path; prefer project-relative data/click
candidate_paths = [
    Path("data/click"),
    Path("../../data/click"),
    Path("../data/click"),
]

data_dir = None
for p in candidate_paths:
    if p.exists() and p.is_dir():
        data_dir = p.resolve()
        break

if data_dir is None:
    raise FileNotFoundError(
        "Could not find dataset folder. Expected data at data/click (or nearby relative paths)."
    )

print(f"Dataset directory: {data_dir}")

# Output path under basic_cnn
output_dir = Path("spatial-tech/models/basic_cnn")
if not output_dir.exists():
    output_dir = Path(".")
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "click_cnn.pth"
meta_path = output_dir / "click_cnn_meta.pth"
print(f"Model will be saved to: {model_path.resolve()}")

Using device: cpu
Dataset directory: /Users/aryamanwade/Desktop/ml_projects/desktop_cnn/spatial-tech/data/click
Model will be saved to: /Users/aryamanwade/Desktop/ml_projects/desktop_cnn/spatial-tech/models/basic_cnn/click_cnn.pth


In [ ]:
# Hyperparameters
IMG_SIZE = 128
BATCH_SIZE = 32
VAL_SPLIT = 0.2
EPOCHS = 10
LEARNING_RATE = 1e-3

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(root=str(data_dir), transform=transform)
class_names = full_dataset.classes
num_classes = len(class_names)

if num_classes < 2:
    raise ValueError("Need at least 2 classes in data/click to train a classifier.")

val_size = max(1, int(len(full_dataset) * VAL_SPLIT))
train_size = len(full_dataset) - val_size

if train_size < 1:
    raise ValueError("Dataset is too small after split. Add more images to data/click.")

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Total images: {len(full_dataset)}")
print(f"Train images: {len(train_dataset)}")
print(f"Val images: {len(val_dataset)}")
print(f"Classes: {class_names}")


class BasicCNN(nn.Module):
    def __init__(self, n_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = BasicCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total

    val_loss, val_acc = evaluate(model, val_loader)

    print(
        f"Epoch [{epoch}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}"
    )

# Save model weights
torch.save(model.state_dict(), model_path)

# Save metadata needed for loading/inference
torch.save(
    {
        "class_names": class_names,
        "img_size": IMG_SIZE,
        "num_classes": num_classes,
        "model_name": "BasicCNN",
    },
    meta_path,
)

print(f"Saved model weights to: {model_path.resolve()}")
print(f"Saved metadata to: {meta_path.resolve()}")